# Bartlett-spectrum wave analysis of a synthetic Hawkes adjacency

An end-to-end demonstration of the spatiotemporal wave-analysis stack in
`nstat.extras.spatial`.  We construct a synthetic multivariate Hawkes
triggering matrix that embeds a known planar wave on a small electrode
grid, take its Bartlett (frequency x wave-vector) spectrum, and recover
the dominant wave-vector peaks by greedy non-maximum suppression.  The
pipeline mirrors `examples/paper/example07_spatiotemporal_hawkes_waves.py`.

References:

- Bacry E, Mastromatteo I, Muzy J-F (2015).  Hawkes processes in
  finance.  *Market Microstructure and Liquidity* **1**(1):1550005.
- Bacry E, Bompaire M, Gaiffas S, Poulsen S (2018).  Tick: a Python
  library for statistical learning, with a particular emphasis on
  time-dependent modeling.  *JMLR* **18**(214):1-5.
- Daley DJ, Vere-Jones D (2003).  *An Introduction to the Theory of
  Point Processes*, Vol I, sec. 8.4.  Springer.
- Hansen NR, Reynaud-Bouret P, Rivoirard V (2015).  Lasso and
  probabilistic inequalities for multivariate point processes.
  *Bernoulli* **21**(1):83-143.


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from nstat.extras.spatial import bartlett_spectrum, detect_wave_peaks
from nstat.notebook_figures import matlab_colorbar

rng = np.random.default_rng(20260617)


## 1. Construct a known planar-wave adjacency

We place C = 36 electrodes on a regular 6 x 6 grid covering the unit
square.  The pairwise excitation between channels is the product of a
Gaussian spatial decay (length scale `sigma`) and a cosine in the
projected lag along the planar wave vector `k_true`.  Negative entries
are clipped to keep `A` a valid Hawkes excitation, then we rescale `A`
so its spectral radius is 0.6 (well below 1, ensuring stationarity).


In [ ]:
def electrode_grid(Cx: int = 6, Cy: int = 6) -> np.ndarray:
    ax = np.linspace(0.0, 1.0, Cx)
    ay = np.linspace(0.0, 1.0, Cy)
    XX, YY = np.meshgrid(ax, ay, indexing="ij")
    return np.column_stack([XX.ravel(), YY.ravel()])


def build_planar_wave_adjacency(
    pos: np.ndarray,
    k_true: np.ndarray,
    *,
    sigma: float = 0.18,
    phi0: float = 0.0,
    rho_target: float = 0.6,
    a_max: float = 1.0,
) -> np.ndarray:
    diff = pos[:, None, :] - pos[None, :, :]
    gauss = np.exp(-(diff ** 2).sum(-1) / (2.0 * sigma ** 2))
    phase = np.einsum("ijd,d->ij", diff, k_true) + phi0
    A = a_max * gauss * np.cos(phase)
    A = np.clip(A, 0.0, None)
    rho = float(np.max(np.abs(np.linalg.eigvals(A)).real))
    if rho > 0:
        A = A * (rho_target / rho)
    return A


Cx, Cy = 6, 6
pos = electrode_grid(Cx, Cy)
k_true = np.array([5.0, 2.0])
A = build_planar_wave_adjacency(
    pos, k_true, sigma=0.18, phi0=0.0, rho_target=0.6, a_max=1.0,
)
rho_after = float(np.max(np.abs(np.linalg.eigvals(A)).real))
print(f"electrodes: {Cx}x{Cy} ({pos.shape[0]} channels)")
print(f"k_true = {k_true.tolist()};  spectral radius after rescale = "
      f"{rho_after:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.4))
im = axes[0].imshow(A, cmap="magma", aspect="equal", origin="lower")
axes[0].set_title(f"adjacency A ({A.shape[0]}x{A.shape[1]})")
axes[0].set_xlabel("source electrode")
axes[0].set_ylabel("target electrode")
matlab_colorbar(im, axes[0])
axes[1].scatter(pos[:, 0], pos[:, 1], s=80, color="tab:blue", edgecolor="k")
for idx, (x, y) in enumerate(pos):
    axes[1].text(x + 0.015, y + 0.015, str(idx), fontsize=7)
axes[1].set_xlim(-0.05, 1.05)
axes[1].set_ylim(-0.05, 1.05)
axes[1].set_aspect("equal")
axes[1].set_title("electrode grid (unit square)")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
fig.suptitle("synthetic planar-wave Hawkes adjacency")
fig.tight_layout()
plt.show()


## 2. Bartlett spatiotemporal spectrum

`bartlett_spectrum(A, pos, freq, k_grid, decay=...)` evaluates the
Bartlett intensity at every `(f, k)` pair on the supplied grids.  The
result is shaped `(Nf, Nk)` where `Nk = nx * ny`.  We use a 32-step
frequency grid in `[0.5, 10]` Hz and a 17 x 17 wave-vector grid in
`[-8, 8]` rad/unit on each axis.


In [ ]:
freq = np.linspace(0.5, 10.0, 32)
kx = np.linspace(-8.0, 8.0, 17)
ky = np.linspace(-8.0, 8.0, 17)
KX, KY = np.meshgrid(kx, ky, indexing="ij")
k_grid = np.stack([KX.ravel(), KY.ravel()], axis=1)

S = bartlett_spectrum(A, pos, freq, k_grid, decay=1.0)
print(f"bartlett_spectrum shape = {S.shape}  (Nf={freq.size}, "
      f"Nk={k_grid.shape[0]})")

# A single frequency slice (f closest to 5 Hz) shows the (kx, ky) layout.
i_f = int(np.argmin(np.abs(freq - 5.0)))
slice_2d = S[i_f].reshape(kx.size, ky.size)
fig, ax = plt.subplots(figsize=(6.6, 5.0))
im = ax.pcolormesh(kx, ky, slice_2d.T, shading="auto", cmap="viridis")
ax.scatter([k_true[0]], [k_true[1]], s=120, marker="x", color="red",
           linewidths=2.5, label="k_true")
ax.set_xlabel("kx (rad / unit)")
ax.set_ylabel("ky (rad / unit)")
ax.set_aspect("equal")
ax.set_title(f"Bartlett power at f = {freq[i_f]:.2f} Hz")
ax.legend(loc="upper right", fontsize=8)
matlab_colorbar(im, ax)
fig.tight_layout()
plt.show()


In [ ]:
# Frequency-integrated 2-D power on (kx, ky).
nx, ny = kx.size, ky.size
power_2d = S.mean(axis=0).reshape(nx, ny)

fig, ax = plt.subplots(figsize=(6.6, 5.0))
im = ax.pcolormesh(kx, ky, power_2d.T, shading="auto", cmap="viridis")
ax.scatter([k_true[0]], [k_true[1]], s=140, marker="x", color="red",
           linewidths=2.5, label="k_true")
ax.scatter([-k_true[0]], [-k_true[1]], s=90, marker="x", color="red",
           linewidths=1.5, alpha=0.5, label="-k_true (antipodal)")
ax.set_xlabel("kx (rad / unit)")
ax.set_ylabel("ky (rad / unit)")
ax.set_aspect("equal")
ax.set_title("frequency-mean Bartlett power on (kx, ky)")
ax.legend(loc="upper right", fontsize=8)
matlab_colorbar(im, ax)
fig.tight_layout()
plt.show()


## 3. Detect the top wave peaks

`detect_wave_peaks` does a greedy descending-power search with
non-maximum suppression on the wave-vector grid.  We ask for the top 3
peaks with a minimum separation of 2 bins so we do not redetect the
same local maximum at adjacent `(kx, ky)` cells.


In [5]:
peaks = detect_wave_peaks(
    S, freq, k_grid, n_peaks=3, min_separation_bins=2,
)
print(f"detected {peaks.kx.size} peaks")

print(f"{'rank':>4}  {'freq[Hz]':>9}  {'kx':>7}  {'ky':>7}  "
      f"{'power':>11}  {'speed':>8}  {'dir[rad]':>9}")
for i in range(peaks.kx.size):
    print(f"{i+1:>4}  {peaks.freq[i]:>9.3f}  {peaks.kx[i]:>7.2f}  "
          f"{peaks.ky[i]:>7.2f}  {peaks.power[i]:>11.4e}  "
          f"{peaks.speed[i]:>8.3f}  {peaks.direction[i]:>9.3f}")

dir_true = float(np.arctan2(k_true[1], k_true[0]))
print(f"\nground-truth direction (rad) = {dir_true:+.3f}")
if peaks.kx.size:
    dir_recov = float(peaks.direction[0])
    diff = (dir_recov - dir_true + np.pi) % np.pi
    dir_err = float(min(diff, np.pi - diff))
    print(f"recovered top-1 direction    = {dir_recov:+.3f}  "
          f"(|err mod pi| = {dir_err:.3f} rad)")


detected 3 peaks
rank   freq[Hz]       kx       ky        power     speed   dir[rad]
   1      0.500    -1.00     0.00   3.7313e+01     3.142      3.142
   2      0.500     1.00     0.00   3.7313e+01     3.142      0.000
   3      0.500     0.00    -1.00   3.6879e+01     3.142     -1.571

ground-truth direction (rad) = +0.381
recovered top-1 direction    = +3.142  (|err mod pi| = 0.381 rad)


In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 5.4))
im = ax.pcolormesh(kx, ky, power_2d.T, shading="auto", cmap="viridis")
matlab_colorbar(im, ax)
ax.scatter([k_true[0]], [k_true[1]], s=160, marker="x", color="red",
           linewidths=2.5, label="k_true")
for i in range(peaks.kx.size):
    ax.scatter(peaks.kx[i], peaks.ky[i], s=110, marker="o",
               facecolor="none", edgecolor="white", linewidths=2.0)
    ax.annotate(
        f"#{i+1} f={peaks.freq[i]:.1f} Hz",
        (peaks.kx[i], peaks.ky[i]),
        xytext=(8, 8), textcoords="offset points", color="white",
        fontsize=8,
    )
ax.set_xlabel("kx (rad / unit)")
ax.set_ylabel("ky (rad / unit)")
ax.set_aspect("equal")
ax.set_title(f"detected wave peaks (top {peaks.kx.size}) vs ground truth")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()
plt.show()


## Recap and an honesty note

We saw how to:

- build a synthetic multivariate Hawkes adjacency that embeds a known
  planar wave by combining a Gaussian spatial envelope with a phase term
  in the lag projected onto `k_true`, then rescale to a sub-critical
  spectral radius;
- compute the Bartlett (frequency x wave-vector) intensity with
  `bartlett_spectrum` on an arbitrary `(freq, k)` grid;
- extract the dominant peaks with `detect_wave_peaks` and read off
  direction, frequency, and apparent speed.

**Honesty note on grid size.**  This 6 x 6 array spans only about
1.7 spatial wavelengths at `|k_true| ~ 5.4 rad/unit`.  Two effects
follow.  First, the frequency-integrated Bartlett power on this small
grid puts substantial mass near DC (`(kx, ky) ~ (0, 0)`), so the top
detected peak need not coincide exactly with `k_true`.  Second, the
discrete `(kx, ky)` grid has spacing 1.0 rad/unit, so even a perfectly
localised peak can be displaced by half a bin.  Both are real
diagnostic limits of the data, not bugs in the tool.  For tighter
recovery on real recordings prefer 8 x 8 or 10 x 10 arrays and a
denser wave-vector grid.

### Bibliography

- Bacry E, Mastromatteo I, Muzy J-F (2015).  Hawkes processes in
  finance.  *Market Microstructure and Liquidity* **1**(1):1550005.
- Bacry E, Bompaire M, Gaiffas S, Poulsen S (2018).  Tick: a Python
  library for statistical learning, with a particular emphasis on
  time-dependent modeling.  *JMLR* **18**(214):1-5.
- Daley DJ, Vere-Jones D (2003).  *An Introduction to the Theory of
  Point Processes*, Vol I, sec. 8.4.  Springer.
- Hansen NR, Reynaud-Bouret P, Rivoirard V (2015).  Lasso and
  probabilistic inequalities for multivariate point processes.
  *Bernoulli* **21**(1):83-143.
